<a href="https://colab.research.google.com/github/DivyaDharshiniG14/Mental-Health-Chatbot/blob/main/mentalhealthchatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pyngrok

In [ ]:
!pip install -q transformers datasets torch streamlit pyngrok accelerate

In [ ]:
from datasets import load_dataset
import pandas as pd
dataset = load_dataset("nbertagnolli/counsel-chat")
print("Example from the dataset:")
print(dataset['train'][10])
df = dataset['train'].to_pandas()
print("\nDataset Info:")
df.info()

In [ ]:
# Cell 3: Preprocess and tokenize the data (Corrected for Batching)
from transformers import AutoTokenizer

# Load the tokenizer for our chosen model
model_name = "microsoft/DialoGPT-medium"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Add a special end-of-string token to our tokenizer
tokenizer.pad_token = tokenizer.eos_token

# Function to format and tokenize our data IN BATCHES
def format_and_tokenize_batch(batch):
    # Create a list of formatted strings from the input batch
    # We zip together the question and answer columns to iterate over them simultaneously
    formatted_texts = [
        f"User: {q}\nBot: {a}{tokenizer.eos_token}"
        for q, a in zip(batch['questionText'], batch['answerText'])
    ]
    # The tokenizer can process a list of strings directly, which is highly efficient
    return tokenizer(
        formatted_texts,
        truncation=True,
        max_length=128,
        padding="max_length"
    )

# Apply the function to the entire dataset, removing rows with missing text first
filtered_dataset = dataset['train'].filter(lambda example: example['questionText'] is not None and example['answerText'] is not None)

# Apply the BATCH-AWARE function to the filtered dataset
tokenized_dataset = filtered_dataset.map(
    format_and_tokenize_batch, # Use the new batch-aware function
    batched=True,              # Keep batching enabled for speed
    batch_size=1000            # Explicitly set the batch size (optional, but good practice)
)

print("\nExample of tokenized data:")
print(tokenized_dataset[10]['input_ids'][:30]) # Show first 30 token IDs

In [ ]:
# Cell 4: Load, Fine-Tune, and Save the Model (Wandb Disabled)
import os
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling

# --- FIX: Add this line to disable wandb ---
os.environ["WANDB_DISABLED"] = "true"

# Load the pre-trained DialoGPT model
model = AutoModelForCausalLM.from_pretrained(model_name)

# Define the directory to save our fine-tuned model
output_dir = "./mental_health_chatbot_model"

# Define training arguments
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=50,
    save_steps=200,
    fp16=True,
    # --- FIX: Add this to explicitly disable reporting to wandb ---
    report_to="none",
)

# A data collator will create batches of data for training
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# Start fine-tuning!
print("🚀 Starting the fine-tuning process...")
trainer.train()
print("✅ Fine-tuning complete!")

# Save the final model and tokenizer
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"✅ Model saved to {output_dir}")

In [ ]:
# Cell 5: Implement helper functions for emotion and crisis detection
from transformers import pipeline

# 1. Emotion Detection
# We use a lightweight, distilled RoBERTa model fine-tuned on emotion data.
print("Loading emotion classification model...")
emotion_classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=1 # Return only the top emotion
)
print("✅ Emotion model loaded.")

def get_emotion(text):
    """Classifies the emotion of a given text."""
    try:
        result = emotion_classifier(text)
        # The pipeline returns a list of lists, so we access it like this
        return result[0][0]['label']
    except Exception as e:
        return "neutral" # Default emotion if classification fails

# 2. Crisis Detection
# This is a simple keyword-based approach for this educational project.
CRISIS_KEYWORDS = [
    "kill myself", "suicide", "want to die", "end my life",
    "self-harm", "hopeless", "can't go on", "no reason to live"
]

def is_crisis(text):
    """Detects crisis intent based on keywords."""
    text_lower = text.lower()
    return any(keyword in text_lower for keyword in CRISIS_KEYWORDS)

# --- Let's test our new functions ---
print("\n--- Testing Helper Functions ---")
test_text_sad = "I feel so alone and I don't know what to do."
test_text_crisis = "I want to end my life, I see no other way out."
test_text_happy = "I finally finished my project and I am so relieved!"

print(f"Text: '{test_text_sad}'")
print(f"Emotion -> {get_emotion(test_text_sad)}")
print(f"Is Crisis? -> {is_crisis(test_text_sad)}")
print("-" * 20)
print(f"Text: '{test_text_crisis}'")
print(f"Emotion -> {get_emotion(test_text_crisis)}")
print(f"Is Crisis? -> {is_crisis(test_text_crisis)}")
print("-" * 20)
print(f"Text: '{test_text_happy}'")
print(f"Emotion -> {get_emotion(test_text_happy)}")
print(f"Is Crisis? -> {is_crisis(test_text_happy)}")
print("-" * 20)

In [ ]:
# Cell 6: Write the Streamlit application code to a file
%%writefile app.py

import streamlit as st
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from collections import deque

# --- PAGE CONFIGURATION ---
st.set_page_config(page_title="MindfulBot", layout="centered", initial_sidebar_state="auto")

# --- MODEL LOADING ---
@st.cache_resource
def load_models():
    """Load all models and tokenizers only once."""
    # 1. Load the fine-tuned conversational model and tokenizer
    model_path = "./mental_health_chatbot_model"
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(model_path)

    # 2. Load the emotion classification pipeline
    emotion_classifier = pipeline(
        "text-classification",
        model="j-hartmann/emotion-english-distilroberta-base",
        top_k=1
    )
    return tokenizer, model, emotion_classifier

tokenizer, model, emotion_classifier = load_models()

# --- SAFETY & HELPER FUNCTIONS ---
CRISIS_KEYWORDS = [
    "kill myself", "suicide", "want to die", "end my life",
    "self-harm", "hopeless", "can't go on", "no reason to live"
]

CRISIS_RESPONSE = """
It sounds like you are in a great deal of pain. Please know that help is available and you don't have to go through this alone.
It's important to talk to someone who can support you right now. Please contact a crisis hotline immediately.

- **USA & Canada:** Call or text 988
- **India:** Call +91 9999666555 (Vandrevala Foundation)
- **United Kingdom:** Call 111
- **Global:** Visit [Befrienders Worldwide](https://www.befrienders.org/) to find a helpline in your country.

Please, reach out to them. They can help.
"""

def is_crisis(text):
    """Simple keyword-based crisis detection."""
    text_lower = text.lower()
    return any(keyword in text_lower for keyword in CRISIS_KEYWORDS)

def get_emotion(text):
    """Get the dominant emotion from text."""
    try:
        return emotion_classifier(text)[0][0]['label']
    except Exception:
        return "neutral"

# --- STREAMLIT UI ---
st.title("🧠 MindfulBot")
st.markdown("Your AI companion for supportive conversations.")
st.markdown("🚨 **Disclaimer:** This is an AI proof-of-concept and not a substitute for a professional therapist. If you are in crisis, please contact the appropriate hotline listed in the sidebar.")

# --- SESSION STATE INITIALIZATION ---
if "history" not in st.session_state:
    st.session_state.history = []
if "memory" not in st.session_state:
    # Use a deque for efficient, fixed-size memory
    st.session_state.memory = deque(maxlen=4)

# --- SIDEBAR ---
with st.sidebar:
    st.header("Settings")
    use_empathetic_response = st.toggle("Enable Empathetic Phrases", value=True)
    if st.button("Clear Conversation History"):
        st.session_state.history = []
        st.session_state.memory.clear()
        st.rerun()
    st.divider()
    st.header("Emergency Hotlines")
    st.markdown(CRISIS_RESPONSE)


# --- CHAT INTERFACE ---
# Display chat messages from history
for message in st.session_state.history:
    with st.chat_message(message["role"]):
        # Add emotion label for user messages
        if message["role"] == "user" and "emotion" in message:
            st.markdown(f"**Emotion:** `{message['emotion']}`\n\n{message['content']}")
        else:
            st.markdown(message["content"])

# --- USER INPUT & MODEL RESPONSE ---
if prompt := st.chat_input("How are you feeling today?"):
    # Add user message to history
    user_emotion = get_emotion(prompt)
    st.session_state.history.append({"role": "user", "content": prompt, "emotion": user_emotion})

    # Display user message
    with st.chat_message("user"):
        st.markdown(f"**Emotion:** `{user_emotion}`\n\n{prompt}")

    # **CRISIS CHECK**
    if is_crisis(prompt):
        with st.chat_message("assistant"):
            st.warning("Crisis Intent Detected")
            st.markdown(CRISIS_RESPONSE)
        st.session_state.history.append({"role": "assistant", "content": CRISIS_RESPONSE})
    else:
        # Construct the input for the model from memory
        memory_prompt = "\n".join(st.session_state.memory) + f"\nUser: {prompt}\nBot:"

        # Tokenize the input
        inputs = tokenizer(memory_prompt, return_tensors="pt")

        # Generate a response from the model
        with st.spinner("Thinking..."):
            output = model.generate(
                inputs["input_ids"],
                max_length=200,
                num_return_sequences=1,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                do_sample=True,
                top_k=40,
                top_p=0.95,
                no_repeat_ngram_size=3,
            )

        # Decode the response
        response = tokenizer.decode(output[0], skip_special_tokens=True)
        # Extract only the bot's part of the response
        bot_response = response.split("Bot:")[-1].strip()

        # Adaptive Response Logic
        empathetic_phrase = ""
        if use_empathetic_response and user_emotion in ["sadness", "fear", "anger", "disgust"]:
             empathetic_phrase = f"I hear that you're feeling {user_emotion}. "

        final_response = empathetic_phrase + bot_response

        # Display bot response
        with st.chat_message("assistant"):
            st.markdown(final_response)

        # Update history and memory
        st.session_state.history.append({"role": "assistant", "content": final_response})
        st.session_state.memory.append(f"User: {prompt}")
        st.session_state.memory.append(f"Bot: {final_response}")

In [ ]:
# Cell 7: Run the Streamlit app using ngrok
from pyngrok import ngrok
import os

# Terminate any existing ngrok tunnels
ngrok.kill()

# Get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "32pwVuytQ0ZjFUbJgIARzM7jO6S_jWEUwmd5m3h5U2KzKXL9"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Open a tunnel to the streamlit port (8501)
public_url = ngrok.connect(8501)
print(f"✅ Your Streamlit app is live at: {public_url}")

# Run the streamlit app in the background
!streamlit run app.py --server.port 8501